# NB07 – Hao hụt và ABC/Pareto kép
Chạy trong thư mục NB07_5_3 trên Colab (đã có `input/`). Đầu ra: `processed/variance_weekly.csv`, các bảng ABC, 2 biểu đồ Pareto và bảng kiểm chứng kịch bản. Nguồn kiểm kê và phiếu hủy là dữ liệu mô phỏng NB03.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
P=Path.cwd(); I=P/'input'; O=P/'processed'; C=P/'charts'
for d in (I,O,C):d.mkdir(exist_ok=True)
inv=pd.read_csv(I/'inventory_weekly.csv'); waste=pd.read_csv(I/'waste_log.csv')
keys=['store_id','ingredient_id','week_start']
assert not inv.duplicated(keys).any()
w=waste.groupby(keys,as_index=False).agg(waste_qty=('waste_qty','sum'),waste_log_value_usd=('waste_value_usd','sum'))
a=inv.merge(w,on=keys,how='left',validate='one_to_one');a[['waste_qty','waste_log_value_usd']]=a[['waste_qty','waste_log_value_usd']].fillna(0)
assert np.allclose(a.recorded_waste,a.waste_qty,atol=.001)
assert np.allclose(a.opening_qty+a.receipt_qty-a.closing_qty,a.actual_usage,atol=.001)
a['variance_qty']=a.actual_usage-a.theoretical_usage
a['variance_rate_pct']=np.where(a.theoretical_usage>0,100*a.variance_qty/a.theoretical_usage,np.nan)
a['waste_rate_pct']=np.where(a.actual_usage>0,100*a.waste_qty/a.actual_usage,np.nan)
a['unexplained_shrinkage_qty']=a.variance_qty-a.waste_qty
a['variance_value_usd']=a.variance_qty*a.unit_cost_usd
a['waste_value_usd']=a.waste_qty*a.unit_cost_usd
a['unexplained_shrinkage_value_usd']=a.unexplained_shrinkage_qty*a.unit_cost_usd
a['positive_shrinkage_value_usd']=a.unexplained_shrinkage_value_usd.clip(lower=0)
a['priority']=np.select([(a.variance_rate_pct>=8)&(a.unexplained_shrinkage_qty>0),(a.variance_rate_pct>=5)&(a.variance_rate_pct<8)&(a.unexplained_shrinkage_qty>0)],['Ưu tiên kiểm tra','Cảnh báo'],default='Bình thường')
a.to_csv(O/'variance_weekly.csv',index=False,encoding='utf-8-sig')
abc=pd.read_csv(O/'abc_revenue.csv')
# ABC by total positive excess usage value; preserve negative amounts separately in variance file.
b=a.groupby(['ingredient_id','ingredient_name','unit'],as_index=False).agg(positive_variance_value_usd=('variance_value_usd',lambda s:s.clip(lower=0).sum()),positive_shrinkage_value_usd=('positive_shrinkage_value_usd','sum'),net_variance_value_usd=('variance_value_usd','sum'),waste_value_usd=('waste_value_usd','sum'))
b=b.sort_values('positive_variance_value_usd',ascending=False).reset_index(drop=True)
total=b.positive_variance_value_usd.sum();b['cumulative_pct']=b.positive_variance_value_usd.cumsum()/total*100;b['previous_pct']=b.cumulative_pct-b.positive_variance_value_usd/total*100
b['abc_class']=np.select([b.previous_pct<80,b.previous_pct<95],['A','B'],default='C');b.to_csv(O/'abc_ingredient_variance.csv',index=False,encoding='utf-8-sig');b.head(10).to_csv(O/'top_10_ingredient_variance.csv',index=False,encoding='utf-8-sig')
def plot(d,val,name,title,file):
 d=d.sort_values(val,ascending=False).reset_index(drop=True);pct=d[val].cumsum()/d[val].sum()*100
 fig,ax=plt.subplots(figsize=(13,5.8));x=np.arange(len(d));ax.bar(x,d[val]);ax.set_ylabel('USD');ax.set_xticks(x);ax.set_xticklabels(d[name],rotation=70,ha='right',fontsize=7);ax.set_title(title)
 ax2=ax.twinx();ax2.plot(x,pct,color='darkorange',lw=2);ax2.axhline(80,color='gray',ls='--');ax2.axhline(95,color='gray',ls=':');ax2.set_ylim(0,105);ax2.set_ylabel('Tỷ trọng lũy kế (%)');fig.tight_layout();fig.savefig(C/file,dpi=180);plt.close(fig)
plot(b,'positive_variance_value_usd','ingredient_name','Pareto giá trị chênh lệch tiêu hao dương theo NVL','abc_ingredient_variance.png')
# Focused tests
s1=a[(a.store_location=="Hell's Kitchen")&(a.ingredient_id=='I008')&a.week_no.between(10,13)].copy()
s2=a[(a.store_location=='Astoria')&(a.ingredient_id=='I006')].copy()
s2other=a[(a.store_location!='Astoria')&(a.ingredient_id=='I006')].copy()
checks=pd.DataFrame([{'scenario':'a','rows':len(s1),'priority_rows':(s1.priority=='Ưu tiên kiểm tra').sum(),'variance_rate_min':s1.variance_rate_pct.min(),'variance_rate_max':s1.variance_rate_pct.max(),'waste_rate_min':s1.waste_rate_pct.min(),'waste_rate_max':s1.waste_rate_pct.max(),'unexplained_qty':s1.unexplained_shrinkage_qty.sum(),'unexplained_usd':s1.unexplained_shrinkage_value_usd.sum()}, {'scenario':'b','rows':len(s2),'warning_rows':(s2.priority=='Cảnh báo').sum(),'priority_rows':(s2.priority=='Ưu tiên kiểm tra').sum(),'variance_rate_weighted':100*s2.variance_qty.sum()/s2.theoretical_usage.sum(),'other_stores_weighted':100*s2other.variance_qty.sum()/s2other.theoretical_usage.sum()}]);checks.to_csv(O/'scenario_validation.csv',index=False,encoding='utf-8-sig')
print('Rows',len(a),'ABC',len(b),'total positive variance value',total)
print('Scenario tests:');print(checks.to_string(index=False));print('Top 10:');print(b[['ingredient_name','positive_variance_value_usd','abc_class']].head(10).to_string(index=False))
print('Summary:',a[['variance_value_usd','waste_value_usd','unexplained_shrinkage_value_usd','positive_shrinkage_value_usd']].sum().to_dict())


**Lưu ý:** `Waste Rate = recorded waste / actual usage`; `Variance Rate = (actual − theoretical)/theoretical`. Kiểm chứng kịch bản không đồng nghĩa kết luận gian lận.